# BDF DE — New ASIN Upload Automation

Turns a Beiersdorf **Listungen** sheet into an upload-ready Amazon Vendor Central template,
plus a QA report that shows where every value came from.

**Scope:** amazon.de only (`de_DE`). Seven product types configured.

Run the cells top to bottom. Steps 1–2 are setup and only need doing once per session.

---
### Before you start

You need two files:

| File | Where it comes from |
|---|---|
| A **blank Vendor Central template** `.xlsm` | Vendor Central → Items → Add Products, with VC language set to **German**. One per product type. |
| A **BDF Listungen** `.xlsx` | Sent by Beiersdorf. Used exactly as received — no manual prep needed. |

The old manual prep steps (padding GTINs with zeros, converting currency columns to numbers,
adding a template column) are all handled in code. Do not pre-process the sheet.


## Step 1 — Install

Colab already has both libraries, but this pins the versions the tool was tested against.


In [ ]:
!pip install -q "openpyxl>=3.1" "PyYAML>=6.0"
print('done')


## Step 2 — Load the project

Pick **one** of the two cells below.

**2A — Google Drive (recommended).** Put the whole `bdf_vc_upload` folder in your Drive.
Your templates and Listungen sheets live in Drive too, so nothing is lost when the session ends.

**2B — Upload the zip.** Faster for a one-off. Everything disappears when the session ends.


In [ ]:
# 2A — Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Adjust if you put the folder somewhere else in Drive.
PROJECT = '/content/drive/MyDrive/bdf_vc_upload'

import os, sys
assert os.path.isdir(PROJECT), f'Not found: {PROJECT} — check the path in Drive'
os.chdir(PROJECT)
sys.path.insert(0, PROJECT)
print('Project:', PROJECT)
print(sorted(os.listdir()))


In [ ]:
# 2B — Upload bdf_vc_upload.zip instead
import os, sys, zipfile
from google.colab import files

up = files.upload()                     # choose bdf_vc_upload.zip
name = next(iter(up))
with zipfile.ZipFile(name) as z:
    z.extractall('/content')

PROJECT = '/content/bdf_vc_upload'
os.chdir(PROJECT)
sys.path.insert(0, PROJECT)
print('Project:', PROJECT)
print(sorted(os.listdir()))


## Step 3 — Point at your two files

If you are on Drive, drop them into the `templates/` and `listungen/` folders and the
cell below will find them. Otherwise run the upload cell after it.


In [ ]:
import glob, os

def pick(folder, pattern, label):
    hits = sorted(g for g in glob.glob(os.path.join(folder, pattern)) if not os.path.basename(g).startswith('~$'))
    if not hits:
        print(f'No {label} found in {folder}/ — use the upload cell below.')
        return None
    for i, h in enumerate(hits):
        print(f'  [{i}] {os.path.basename(h)}')
    return hits

print('Templates in templates/:')
TEMPLATES = pick('templates', '*.xlsm', 'template')
print('\nListungen sheets in listungen/:')
SHEETS = pick('listungen', '*.xlsx', 'Listungen sheet')


In [ ]:
# Optional — upload the two files directly instead of using the folders
from google.colab import files
import shutil, os

os.makedirs('templates', exist_ok=True)
os.makedirs('listungen', exist_ok=True)

print('Choose the blank Vendor Central .xlsm template:')
for name in files.upload():
    shutil.move(name, os.path.join('templates', name))

print('Choose the BDF Listungen .xlsx:')
for name in files.upload():
    shutil.move(name, os.path.join('listungen', name))

print('\nUploaded. Re-run the cell above to list them.')


## Step 4 — Choose which files to use

Set the two indexes from the lists printed above.

`FILTER_TEMPLATE` matters when one Listungen sheet covers several product types. Set it to the
value in the sheet's `Template` column — e.g. `'Body Deodorant'`, `'Hair Styling Agent'` — so only
the matching rows go into this template. Leave it `None` to use every row.


In [ ]:
TEMPLATE_INDEX   = 0
SHEET_INDEX      = 0
FILTER_TEMPLATE  = None      # e.g. 'Body Deodorant'
OUT_DIR          = 'output'

TEMPLATE_PATH = TEMPLATES[TEMPLATE_INDEX]
SHEET_PATH    = SHEETS[SHEET_INDEX]
print('Template :', TEMPLATE_PATH)
print('Input    :', SHEET_PATH)
print('Filter   :', FILTER_TEMPLATE)


### Not sure what to put in `FILTER_TEMPLATE`?

This shows what is actually in the sheet's `Template` column.


In [ ]:
from bdfvc.inputsheet import ListungenSheet
from collections import Counter

sheet = ListungenSheet(SHEET_PATH)
print(f'Header row {sheet.header_row}, {len(sheet.columns)} columns recognised')

counts = Counter()
for _, rec in sheet.rows():
    counts[rec.get('template_hint') or '(blank)'] += 1

print('\nProducts per Template value:')
for value, n in counts.most_common():
    print(f'  {n:4}  {value}')

if sheet.unmapped:
    print(f'\n{len(sheet.unmapped)} column(s) ignored (not needed for the upload):')
    print('  ' + ', '.join(sheet.unmapped))


## Step 5 — Run it


In [ ]:
from fill_bdf import build

report, out_xlsm, out_qa = build(
    TEMPLATE_PATH,
    SHEET_PATH,
    OUT_DIR,
    filter_template=FILTER_TEMPLATE,
)

print(f'Product type : {report.template.product_type} ({report.template.locale})')
print(f'Products     : {len(report.rows)}')
print(f'Upload file  : {out_xlsm}')
print(f'QA report    : {out_qa}')
print()
print('STATUS:', report.status)


## Step 6 — Read the QA result

**Blockers** must be fixed before uploading. **Review items** are things a human should confirm;
the file can still go up.

`scent` is always a review item by design — the historical uploads are inconsistent on it, so the
tool picks deterministically and asks you to confirm rather than guessing silently.


In [ ]:
blockers = report.blockers()
warnings = report.warnings()

if blockers:
    print(f'{len(blockers)} BLOCKING issue(s) — do not upload yet:')
    for b in blockers:
        print('  -', b)
else:
    print('No blocking issues. Every required column is filled and every value is a valid template option.')

print()
if warnings:
    print(f'{len(warnings)} item(s) to review:')
    for w in warnings:
        print('  -', w)
else:
    print('Nothing flagged for review.')


### Where each value came from

One row per populated cell. Useful when a value looks wrong and you want to know whether it came
from the source sheet, a fixed default, a lookup table or a rule.


In [ ]:
import pandas as pd

rows = []
for r in report.rows:
    for f in r.fields.values():
        if f.status == 'absent':
            continue
        rows.append({
            'row': r.target_row, 'NART': r.nart, 'field': f.code,
            'column': f.label, 'requirement': f.requirement,
            'value': f.value, 'source': f.provenance,
            'status': f.status, 'note': f.note,
        })

df = pd.DataFrame(rows)
print(f'{len(df)} cells written across {len(report.rows)} products')
print()
print(df.groupby('source').size().to_string())

# Anything that is not a clean 'ok'
df[df.status != 'ok'][['row', 'NART', 'column', 'value', 'status', 'note']]


## Step 7 — Download

On Drive both files are already saved in `output/`. This cell downloads them to your machine.


In [ ]:
from google.colab import files
files.download(str(out_xlsm))
files.download(str(out_qa))


---
## Appendix A — Check the tool still works

Only runs if the six verified historical uploads are present. Regenerates each one from its own
source sheet and diffs cell by cell. Expected result: **98.1% match, 0 blockers**.

Worth running after anyone edits the config.


In [ ]:
import os
if os.path.isdir('/mnt/user-data/uploads'):
    !python tools/regress.py
else:
    print('Reference uploads not available in this environment — skipping.')
    print('To run this, place the six verified .xlsm uploads and three Listungen sheets')
    print('in a folder and update UP in tools/regress.py.')


## Appendix B — Run several product types in one go

When a Listungen sheet covers multiple templates, loop over them. Each template still needs its
own blank `.xlsm` in `templates/`.


In [ ]:
# Map each Template value in the sheet to its blank template file
BATCH = {
    # 'Body Deodorant':     'templates/Deodorants_2026-08-11.xlsm',
    # 'Hair Styling Agent': 'templates/Haarstyling-Mittel_2026-08-11.xlsm',
}

results = []
for filter_value, template_file in BATCH.items():
    rep, xlsm, qa = build(template_file, SHEET_PATH, OUT_DIR, filter_template=filter_value)
    results.append((filter_value, len(rep.rows), rep.status, len(rep.blockers()), len(rep.warnings())))
    print(f'{filter_value:24} {len(rep.rows):3} products   {rep.status}')

if results:
    import pandas as pd
    display(pd.DataFrame(results, columns=['Template', 'Products', 'Status', 'Blockers', 'Review']))
else:
    print('Fill in BATCH above to use this cell.')
